## Invoice DocAI v2 -- 03 Donut Inference

Run the OCR-free Donut model on SROIE validation data (347 docs).

### Plan
1. Setup: resolve PROJECT_ROOT, load manifest
2. Load Donut model (`philschmid/donut-base-sroie` preferred)
3. Improved SROIE parser (tag-based + token2json + regex fallbacks)
4. Inference with checkpointing every 20 docs
5. Metrics and comparison with OCR baseline
6. Diagnostic: raw Donut output for 5 sample documents

In [ ]:
from pathlib import Path
import json
import re
import sys
import time
from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

# Mount Google Drive in Colab (if available)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ---------------------------------------------------------------------------
# Resolve PROJECT_ROOT (invoice_docai)
# ---------------------------------------------------------------------------
def is_project_root(p: Path) -> bool:
    return (
        (p / 'data' / 'sroie' / 'processed').exists()
        and (p / 'v2' / 'src').exists()
    )

def resolve_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd / 'invoice_docai',
        Path('/content/drive/MyDrive/ORC/invoice_docai'),
        Path('/content/drive/MyDrive/invoice_docai'),
        Path('/content/drive/MyDrive/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/drive/MyDrive/Yandex.Disk/Yandex.Disk/ML Neimark/From OCR 2 Transformers/invoice_docai'),
        Path('/content/invoice_docai'),
        Path(r'c:\Yandex.Disk\Yandex.Disk\ML Neimark\From OCR 2 Transformers\invoice_docai'),
    ]
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if is_project_root(p):
            return p
        child = p / 'invoice_docai'
        if child.exists() and is_project_root(child):
            return child

    for base in [Path('/content/drive/MyDrive'), Path('/content/drive'), Path('/content'), cwd, cwd.parent]:
        if not base.exists():
            continue
        try:
            for pattern in ['invoice_docai', 'invoice_docai*', '*invoice*doc*ai*']:
                for p in base.rglob(pattern):
                    if p.is_dir() and is_project_root(p):
                        return p.resolve()
        except Exception:
            pass
    return None

PROJECT_ROOT = resolve_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate invoice_docai project with data/sroie/processed and v2/src')

V2_SRC = PROJECT_ROOT / 'v2' / 'src'
if V2_SRC.exists() and str(V2_SRC) not in sys.path:
    sys.path.insert(0, str(V2_SRC))

# Also add legacy src/ for shared utilities
LEGACY_SRC = PROJECT_ROOT / 'src'
if LEGACY_SRC.exists() and str(LEGACY_SRC) not in sys.path:
    sys.path.insert(0, str(LEGACY_SRC))

try:
    from docai_utils import normalize_date, normalize_total, normalize_vendor, evaluate
    print('Loaded docai_utils from', V2_SRC if (V2_SRC / 'docai_utils.py').exists() else LEGACY_SRC)
except ImportError:
    print('WARNING: docai_utils not found, using inline fallback functions')

    DATE_PATTERNS = [
        re.compile(r'\b(\d{2})[./-](\d{2})[./-](\d{4})\b'),
        re.compile(r'\b(\d{4})[./-](\d{2})[./-](\d{2})\b'),
    ]
    MONEY_PATTERN = re.compile(r'[-+]?\d+[\d\s.,]*\d')

    def normalize_total(value):
        if value is None: return ''
        text = str(value).strip()
        if not text: return ''
        text = text.replace(' ', '').replace('$', '').replace('EUR', '').replace('RM', '')
        text = text.replace(',', '.')
        match = MONEY_PATTERN.search(text)
        if not match: return ''
        candidate = match.group(0)
        if candidate.count('.') > 1:
            parts = candidate.split('.')
            candidate = ''.join(parts[:-1]) + '.' + parts[-1]
        try:
            return f'{Decimal(candidate):.2f}'
        except InvalidOperation:
            return ''

    def normalize_date(value):
        if value is None: return ''
        text = str(value).strip()
        if not text: return ''
        for pattern in DATE_PATTERNS:
            match = pattern.search(text)
            if not match: continue
            a, b, c = match.groups()
            if len(a) == 4: year, month, day = a, b, c
            else: day, month, year = a, b, c
            try: return f'{int(year):04d}-{int(month):02d}-{int(day):02d}'
            except ValueError: continue
        return ''

    def normalize_vendor(value):
        if value is None: return ''
        text = str(value).strip().lower()
        return re.sub(r'\s+', ' ', text)

    def compute_field_metrics(y_true, y_pred):
        tp = fp = fn = 0
        for t, p in zip(y_true, y_pred):
            t_ok = bool(str(t).strip()); p_ok = bool(str(p).strip())
            if p_ok and t_ok and str(p).strip() == str(t).strip(): tp += 1
            elif p_ok and (not t_ok or str(p).strip() != str(t).strip()): fp += 1
            elif t_ok and not p_ok: fn += 1
            elif t_ok and p_ok and str(p).strip() != str(t).strip(): fp += 1; fn += 1
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        return {'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

    def evaluate(ground_truth_df, predictions_df):
        merged = ground_truth_df.merge(predictions_df, on='id', how='inner')
        for col in ['gt_vendor', 'pred_vendor']:
            merged[col] = merged[col].fillna('').map(normalize_vendor)
        for col in ['gt_date', 'pred_date']:
            merged[col] = merged[col].fillna('').map(normalize_date)
        for col in ['gt_total', 'pred_total']:
            merged[col] = merged[col].fillna('').map(normalize_total)
        metrics_rows = []
        total_tp = total_fp = total_fn = 0
        for field in ['vendor', 'date', 'total']:
            fm = compute_field_metrics(merged[f'gt_{field}'].tolist(), merged[f'pred_{field}'].tolist())
            exact = (merged[f'gt_{field}'] == merged[f'pred_{field}']).mean() if len(merged) else 0.0
            fm['field'] = field; fm['exact_match'] = exact
            metrics_rows.append(fm)
            total_tp += fm['tp']; total_fp += fm['fp']; total_fn += fm['fn']
        micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
        micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
        micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0
        metrics_rows.append({'field': 'micro', 'precision': micro_p, 'recall': micro_r,
                             'f1': micro_f1, 'exact_match': np.nan,
                             'tp': total_tp, 'fp': total_fp, 'fn': total_fn})
        metrics_df = pd.DataFrame(metrics_rows)
        errors = merged[
            (merged['gt_vendor'] != merged['pred_vendor'])
            | (merged['gt_date'] != merged['pred_date'])
            | (merged['gt_total'] != merged['pred_total'])
        ].copy()
        errors['num_wrong_fields'] = (
            (errors['gt_vendor'] != errors['pred_vendor']).astype(int)
            + (errors['gt_date'] != errors['pred_date']).astype(int)
            + (errors['gt_total'] != errors['pred_total']).astype(int)
        )
        errors = errors.sort_values('num_wrong_fields', ascending=False)
        return metrics_df, errors

PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'sroie' / 'processed'
OUTPUT_ROOT = PROJECT_ROOT / 'v2' / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(PROCESSED_ROOT / 'manifest_val.csv')
manifest = manifest.dropna(subset=['image_path']).reset_index(drop=True)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_ROOT  = {OUTPUT_ROOT}')
print(f'Loaded {len(manifest)} validation samples')
manifest.head(3)

## 2. Load Donut Model

Try SROIE-finetuned checkpoints first, fall back to CORD.

In [ ]:
# Install dependencies if needed
try:
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    print('transformers already installed')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'transformers', 'sentencepiece', 'accelerate', '-q'])
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    print('transformers installed')

# Prefer SROIE-finetuned models, then CORD
MODEL_CANDIDATES = [
    'philschmid/donut-base-sroie',
    'Assadullah/donut-base-sroie',
    'naver-clova-ix/donut-base-finetuned-cord-v2',
]

model_loaded = False
last_error = None

for name in MODEL_CANDIDATES:
    try:
        processor = DonutProcessor.from_pretrained(name)
        model = VisionEncoderDecoderModel.from_pretrained(name)
        MODEL_NAME = name
        model_loaded = True
        break
    except Exception as e:
        last_error = e
        continue

if not model_loaded:
    raise RuntimeError(f'Failed to load any Donut model. Last error: {last_error}')

# Task prompt depends on model fine-tuning recipe
if 'sroie' in MODEL_NAME.lower():
    TASK_PROMPT = '<s_sroie>'
elif 'cord' in MODEL_NAME.lower():
    TASK_PROMPT = '<s_cord-v2>'
else:
    TASK_PROMPT = '<s_cord-v2>'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

print(f'Device:      {device}')
print(f'Model:       {MODEL_NAME}')
print(f'Task prompt: {TASK_PROMPT}')

## 3. Improved SROIE Parser

Multi-layer parsing strategy:
1. Extract from SROIE-specific closing tags (`</s_company>`, `</s_date>`, `</s_total>`, `</s_address>`)
2. Fall back to `processor.token2json()`
3. Fall back to CORD-style tag extraction
4. Fall back to regex on cleaned text

In [ ]:
# ---------------------------------------------------------------------------
# SROIE tag patterns
# ---------------------------------------------------------------------------
SROIE_TAG_RE = {
    'company': re.compile(r'<s_company>\s*(.*?)\s*</s_company>', re.DOTALL),
    'date':    re.compile(r'<s_date>\s*(.*?)\s*</s_date>', re.DOTALL),
    'total':   re.compile(r'<s_total>\s*(.*?)\s*</s_total>', re.DOTALL),
    'address': re.compile(r'<s_address>\s*(.*?)\s*</s_address>', re.DOTALL),
}

# CORD tag patterns (fallback)
CORD_TAG_VALUE_RE = re.compile(r'<(?P<tag>[^>]+)>(?P<val>.*?)</[^>]+>')

# Known key aliases for JSON extraction
KEY_MAPS = {
    'vendor': {'vendor', 'company', 'store', 'seller', 'merchant'},
    'date':   {'date', 'invoice_date', 'issuedate'},
    'total':  {'total', 'total_price', 'amount', 'grand_total', 'totalamount'},
}

DATE_REGEX = re.compile(r'\b(\d{2}[./-]\d{2}[./-]\d{4}|\d{4}[./-]\d{2}[./-]\d{2})\b')
MONEY_2DP_RE = re.compile(r'\b(\d{1,6}\.\d{2})\b')


def _find_value_in_dict(payload, candidates):
    """Recursively find a value in nested dict/list by key name."""
    if isinstance(payload, dict):
        for key, val in payload.items():
            key_l = str(key).lower().replace(' ', '').replace('-', '_')
            if key_l in candidates:
                return val
            nested = _find_value_in_dict(val, candidates)
            if nested not in [None, '']:
                return nested
    if isinstance(payload, list):
        for item in payload:
            nested = _find_value_in_dict(item, candidates)
            if nested not in [None, '']:
                return nested
    return ''


def parse_sroie_donut_output(raw_text: str) -> dict:
    """Multi-layer parser for Donut output on SROIE data.

    Strategy:
      1. Try SROIE-specific tags (</s_company>, </s_date>, </s_total>)
      2. Fall back to processor.token2json()
      3. Fall back to CORD-style tag extraction
      4. Fall back to regex on cleaned text

    Returns dict with pred_vendor, pred_date, pred_total.
    """
    vendor = ''
    date = ''
    total = ''

    # --- Layer 1: SROIE-specific tags ---
    m_company = SROIE_TAG_RE['company'].search(raw_text)
    m_date    = SROIE_TAG_RE['date'].search(raw_text)
    m_total   = SROIE_TAG_RE['total'].search(raw_text)

    if m_company:
        vendor = m_company.group(1).strip()
    if m_date:
        date = normalize_date(m_date.group(1).strip())
    if m_total:
        total = normalize_total(m_total.group(1).strip())

    # If all three found via tags, return early
    if vendor and date and total:
        return {'pred_vendor': vendor, 'pred_date': date, 'pred_total': total}

    # --- Layer 2: token2json() ---
    if not all([vendor, date, total]):
        try:
            payload = processor.token2json(raw_text)
            if isinstance(payload, dict):
                if not vendor:
                    v = _find_value_in_dict(payload, KEY_MAPS['vendor'])
                    if v:
                        vendor = str(v).strip()
                if not date:
                    d = _find_value_in_dict(payload, KEY_MAPS['date'])
                    if d:
                        date = normalize_date(str(d))
                if not total:
                    t = _find_value_in_dict(payload, KEY_MAPS['total'])
                    if t:
                        total = normalize_total(str(t))
        except Exception:
            pass

    # --- Layer 3: CORD-style tag extraction ---
    if not vendor:
        nm_matches = re.findall(r'<s_nm>\s*([^<]+)\s*</s_nm>', raw_text)
        for cand in nm_matches:
            cand = cand.strip()
            letters = sum(ch.isalpha() for ch in cand)
            digits = sum(ch.isdigit() for ch in cand)
            if letters >= 3 and letters >= digits and len(cand) >= 4:
                vendor = cand
                break

    if not total:
        total_candidates = (
            re.findall(r'<s_total_price>\s*([^<]+)\s*</s_total_price>', raw_text)
            + re.findall(r'<s_total>\s*([^<]+)\s*</s_total>', raw_text)
        )
        for val in total_candidates:
            t = normalize_total(val.strip())
            if t:
                total = t
                break

    if not date:
        cashprice_dates = re.findall(r'<s_cashprice>\s*([^<]+)\s*</s_cashprice>', raw_text)
        for val in cashprice_dates:
            d = normalize_date(val.strip())
            if d:
                date = d
                break

    # --- Layer 4: Regex fallback on cleaned text ---
    clean_text = re.sub(r'<[^>]+>', ' ', raw_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    if not date:
        m = DATE_REGEX.search(clean_text)
        if m:
            date = normalize_date(m.group(1))

    if not total:
        # Find all monetary values (X.XX format), pick the largest reasonable one
        money_matches = MONEY_2DP_RE.findall(clean_text)
        candidates = []
        for m in money_matches:
            try:
                val = float(m)
                if 0.01 <= val <= 50000:
                    candidates.append((val, normalize_total(m)))
            except ValueError:
                pass
        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            total = candidates[0][1]

    if not vendor:
        # Take the first text chunk that looks like a company name
        words = clean_text.split()
        if len(words) >= 2:
            # Take first ~5 words if they have enough letters
            snippet = ' '.join(words[:5])
            letters = sum(ch.isalpha() for ch in snippet)
            if letters >= 3:
                vendor = snippet

    return {'pred_vendor': vendor, 'pred_date': date, 'pred_total': total}


print('parse_sroie_donut_output() defined')

## 4. Donut Inference Function

In [ ]:
def donut_predict(image_path: str) -> dict:
    """Run Donut inference on a single image.

    Returns dict with pred_vendor, pred_date, pred_total, raw_text.
    """
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)

    decoder_input_ids = processor.tokenizer(
        TASK_PROMPT,
        add_special_tokens=False,
        return_tensors='pt',
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=model.decoder.config.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    seq = processor.batch_decode(outputs.sequences)[0]
    # Remove EOS / PAD / leading task prompt
    seq = seq.replace(processor.tokenizer.eos_token, '')
    seq = seq.replace(processor.tokenizer.pad_token, '')
    seq = re.sub(r'^<.*?>', '', seq, count=1).strip()

    parsed = parse_sroie_donut_output(seq)
    parsed['raw_text'] = seq
    return parsed


# Quick sanity test on first document
test_pred = donut_predict(manifest.iloc[0]['image_path'])
print('Sanity test on first doc:')
for k, v in test_pred.items():
    if k == 'raw_text':
        print(f'  {k}: {v[:120]}...')
    else:
        print(f'  {k}: {v!r}')

## 5. Run Inference on All 347 Validation Documents

Checkpointing every 20 docs to `outputs/donut_predictions_clean.csv`.

In [ ]:
CHECKPOINT_EVERY = 20
OUTPUT_CSV = OUTPUT_ROOT / 'donut_predictions_clean.csv'

# Resume from checkpoint if exists
if OUTPUT_CSV.exists():
    existing_df = pd.read_csv(OUTPUT_CSV)
    done_ids = set(existing_df['id'].tolist())
    rows = existing_df.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} docs already done')
else:
    done_ids = set()
    rows = []

latencies = []

remaining = manifest[~manifest['id'].isin(done_ids)].reset_index(drop=True)
print(f'Remaining docs to process: {len(remaining)}')

for i, row in tqdm(remaining.iterrows(), total=len(remaining), desc='Donut inference'):
    start = time.perf_counter()
    try:
        pred = donut_predict(row['image_path'])
    except Exception as e:
        print(f'\nERROR on {row["id"]}: {e}')
        pred = {'pred_vendor': '', 'pred_date': '', 'pred_total': '', 'raw_text': f'ERROR: {e}'}
    elapsed_ms = (time.perf_counter() - start) * 1000
    latencies.append(elapsed_ms)

    rows.append({
        'id': row['id'],
        'pred_vendor': pred['pred_vendor'],
        'pred_date': pred['pred_date'],
        'pred_total': pred['pred_total'],
    })

    # Checkpoint every N docs
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)

# Final save
pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUTPUT_CSV, index=False)

print(f'\nSaved {len(pred_df)} predictions to {OUTPUT_CSV}')
if latencies:
    print(f'Latency mean: {np.mean(latencies):.0f} ms/doc')
    print(f'Latency p50:  {np.percentile(latencies, 50):.0f} ms/doc')
    print(f'Latency p90:  {np.percentile(latencies, 90):.0f} ms/doc')
pred_df.head(5)

## 6. Metrics and Comparison

In [ ]:
# Load predictions (in case this cell is run standalone)
if 'pred_df' not in dir() or pred_df is None:
    pred_df = pd.read_csv(OUTPUT_CSV)

gt_df = manifest[['id', 'gt_vendor', 'gt_date', 'gt_total']].copy()
metrics_df, errors_df = evaluate(gt_df, pred_df)

print('=== Donut Inference Metrics ===')
print(f'Model: {MODEL_NAME}')
print(f'Docs evaluated: {len(pred_df)}\n')
display(metrics_df)

print(f'\nTop errors ({min(15, len(errors_df))} shown):')
display(errors_df[['id', 'gt_vendor', 'pred_vendor', 'gt_date', 'pred_date',
                    'gt_total', 'pred_total', 'num_wrong_fields']].head(15))

In [ ]:
# Compare with OCR baseline if available
ocr_pred_path = PROJECT_ROOT / 'v2' / 'outputs' / 'ocr_predictions_clean.csv'
if not ocr_pred_path.exists():
    ocr_pred_path = PROJECT_ROOT / 'outputs' / 'ocr_predictions_clean.csv'

if ocr_pred_path.exists():
    ocr_pred_df = pd.read_csv(ocr_pred_path)
    ocr_metrics, _ = evaluate(gt_df, ocr_pred_df)

    print('=== Comparison: Donut vs OCR Baseline ===')
    comparison = pd.DataFrame({
        'field': metrics_df['field'],
        'donut_f1': metrics_df['f1'].values,
        'donut_precision': metrics_df['precision'].values,
        'donut_recall': metrics_df['recall'].values,
        'ocr_f1': ocr_metrics['f1'].values,
        'ocr_precision': ocr_metrics['precision'].values,
        'ocr_recall': ocr_metrics['recall'].values,
    })
    display(comparison)
else:
    print(f'OCR baseline predictions not found at {ocr_pred_path}')
    print('Run notebook 02 first to generate OCR baseline predictions.')

In [ ]:
import matplotlib.pyplot as plt

# Bar chart: field-level metrics
field_only = metrics_df[metrics_df['field'].isin(['vendor', 'date', 'total'])].copy()
plot_data = field_only.set_index('field')[['precision', 'recall', 'f1']]

ax = plot_data.plot(kind='bar', figsize=(9, 4), ylim=(0, 1), rot=0)
ax.set_title(f'Donut Inference: field-level metrics ({MODEL_NAME})')
ax.set_ylabel('Score')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

# Micro F1 summary
micro = metrics_df[metrics_df['field'] == 'micro'].iloc[0]
print(f'Micro F1: {micro["f1"]:.3f} | P: {micro["precision"]:.3f} | R: {micro["recall"]:.3f}')

## 7. Diagnostic: Raw Donut Output for 5 Sample Documents

Inspect what Donut actually produces to understand its output format.

In [ ]:
diag_n = 5
diag_subset = manifest.sample(min(diag_n, len(manifest)), random_state=7).reset_index(drop=True)

print(f'Diagnostic: raw Donut output for {len(diag_subset)} sample documents\n')

for i, row in diag_subset.iterrows():
    pred = donut_predict(row['image_path'])
    raw = pred.get('raw_text', '')

    # Try token2json for structured view
    try:
        structured = processor.token2json(raw)
    except Exception:
        structured = {}

    print('=' * 80)
    print(f'Doc ID:      {row["id"]}')
    print(f'GT vendor:   {row["gt_vendor"]}')
    print(f'GT date:     {row["gt_date"]}')
    print(f'GT total:    {row["gt_total"]}')
    print(f'Pred vendor: {pred["pred_vendor"]}')
    print(f'Pred date:   {pred["pred_date"]}')
    print(f'Pred total:  {pred["pred_total"]}')
    print(f'token2json keys: {list(structured.keys()) if isinstance(structured, dict) else type(structured)}')
    print(f'Raw text (first 600 chars):')
    print(raw[:600] if raw else '<EMPTY>')
    print()

## Summary

This notebook ran Donut inference on all SROIE validation documents with an improved
multi-layer parser. Predictions are saved to `v2/outputs/donut_predictions_clean.csv`
and can be compared with the OCR baseline and with fine-tuned Donut (notebook 03b).